In [ ]:
import torch
print(torch.__version__)

In [1]:
from torch.utils.data import Dataset
import PIL.Image as Image
import os

def make_dataset(root):
    
    imgs = []
    
    ori_path = os.path.join(root, "Data")
    
    ground_path = os.path.join(root, "Ground")

    names = os.listdir(ori_path)
    n = len(names)
    # Read the corresponding image and label folder paths and put them into a list
    for i in range(n):
        img = os.path.join(ori_path, names[i])
        mask = os.path.join(ground_path, names[i])
        imgs.append((img, mask))
    return imgs

# Inherit from the Dataset class
class LiverDataset(Dataset):
    def __init__(self, root, transform=None, target_transform=None):
        # Get the file paths of the dataset (img, mask)
        imgs = make_dataset(root)
        self.imgs = imgs
        self.transform = transform
        self.target_transform = target_transform

    def __getitem__(self, index):
        # Get the image and label paths
        x_path, y_path = self.imgs[index]
        img_x = Image.open(x_path).convert('L')
        img_y = Image.open(y_path).convert('L')
        if self.transform is not None:
            img_x = self.transform(img_x)
        if self.target_transform is not None:
            img_y = self.target_transform(img_y)
        return img_x, img_y

    def __len__(self):
        return len(self.imgs)

In [2]:
import torch
from torch import nn
import torch.nn.functional as F
from einops import rearrange
import math

# ===== Simplified Fusion Module =====
class SimpleFuse(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=False)
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)

class DepthWiseConv2d(nn.Module):
    def __init__(self, dim_in, dim_out, kernel_size=3, padding=1, stride=1, dilation=1):
        super().__init__()
        self.conv1 = nn.Conv2d(dim_in, dim_in, kernel_size=kernel_size,
                               padding=padding, stride=stride,
                               dilation=dilation, groups=dim_in)
        self.norm_layer = nn.GroupNorm(4, dim_in)
        self.conv2 = nn.Conv2d(dim_in, dim_out, kernel_size=1)

    def forward(self, x):
        return self.conv2(self.norm_layer(self.conv1(x)))

class LayerNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-6, data_format="channels_last"):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(normalized_shape))
        self.bias = nn.Parameter(torch.zeros(normalized_shape))
        self.eps = eps
        self.data_format = data_format
        self.normalized_shape = (normalized_shape, )

    def forward(self, x):
        if self.data_format == "channels_last":
            return F.layer_norm(x, self.normalized_shape, self.weight, self.bias, self.eps)
        else:
            u = x.mean(1, keepdim=True)
            s = (x - u).pow(2).mean(1, keepdim=True)
            x = (x - u) / torch.sqrt(s + self.eps)
            return self.weight[:, None, None] * x + self.bias[:, None, None]

class Grouped_multi_Attention(nn.Module):
    def __init__(self, dim_in, dim_out, x=8, y=8):
        super().__init__()

        c_dim_in = dim_in//4
        k_size=3
        pad=(k_size-1) // 2

        self.params_xy = nn.Parameter(torch.Tensor(1, c_dim_in, x, y), requires_grad=True)
        nn.init.ones_(self.params_xy)
        self.conv_xy = nn.Sequential(nn.Conv2d(c_dim_in, c_dim_in, kernel_size=k_size, padding=pad, groups=c_dim_in), nn.GELU(), nn.Conv2d(c_dim_in, c_dim_in, 1))

        self.params_zx = nn.Parameter(torch.Tensor(1, 1, c_dim_in, x), requires_grad=True)
        nn.init.ones_(self.params_zx)
        self.conv_zx = nn.Sequential(nn.Conv1d(c_dim_in, c_dim_in, kernel_size=k_size, padding=pad, groups=c_dim_in), nn.GELU(), nn.Conv1d(c_dim_in, c_dim_in, 1))

        self.params_zy = nn.Parameter(torch.Tensor(1, 1, c_dim_in, y), requires_grad=True)
        nn.init.ones_(self.params_zy)
        self.conv_zy = nn.Sequential(nn.Conv1d(c_dim_in, c_dim_in, kernel_size=k_size, padding=pad, groups=c_dim_in), nn.GELU(), nn.Conv1d(c_dim_in, c_dim_in, 1))

        self.dw = nn.Sequential(
                nn.Conv2d(c_dim_in, c_dim_in, 1),
                nn.GELU(),
                nn.Conv2d(c_dim_in, c_dim_in, kernel_size=3, padding=1, groups=c_dim_in)
        )

        self.norm1 = LayerNorm(dim_in, eps=1e-6, data_format='channels_first')
        self.norm2 = LayerNorm(dim_in, eps=1e-6, data_format='channels_first')

        self.ldw = nn.Sequential(
                nn.Conv2d(dim_in, dim_in, kernel_size=3, padding=1, groups=dim_in),
                nn.GELU(),
                nn.Conv2d(dim_in, dim_out, 1),
        )

    def forward(self, x):
        x = self.norm1(x)
        x1, x2, x3, x4 = torch.chunk(x, 4, dim=1)
        B, C, H, W = x1.size()
        #----------xy----------#
        params_xy = self.params_xy
        x1 = x1 * self.conv_xy(F.interpolate(params_xy, size=x1.shape[2:4],mode='bilinear', align_corners=True))
        #----------zx----------#
        x2 = x2.permute(0, 3, 1, 2)
        params_zx = self.params_zx
        x2 = x2 * self.conv_zx(F.interpolate(params_zx, size=x2.shape[2:4],mode='bilinear', align_corners=True).squeeze(0)).unsqueeze(0)
        x2 = x2.permute(0, 2, 3, 1)
        #----------zy----------#
        x3 = x3.permute(0, 2, 1, 3)
        params_zy = self.params_zy
        x3 = x3 * self.conv_zy(F.interpolate(params_zy, size=x3.shape[2:4],mode='bilinear', align_corners=True).squeeze(0)).unsqueeze(0)
        x3 = x3.permute(0, 2, 1, 3)
        #----------dw----------#
        x4 = self.dw(x4)
        #----------concat----------#
        x = torch.cat([x1,x2,x3,x4],dim=1)
        #----------ldw----------#
        x = self.norm2(x)
        x = self.ldw(x)
        return x



class improved_UNet(nn.Module):
    def __init__(self, num_classes=1, input_channels=1, c_list=[8,16,24,32,48,64]):
        super().__init__()
        # Encoder
        self.encoder1 = nn.Conv2d(input_channels, c_list[0], 3, padding=1)
        self.encoder2 = nn.Conv2d(c_list[0], c_list[1], 3, padding=1)
        self.encoder3 = nn.Conv2d(c_list[1], c_list[2], 3, padding=1)
        self.encoder4 = Grouped_multi_Attention(c_list[2], c_list[3])
        self.encoder5 = Grouped_multi_Attention(c_list[3], c_list[4])
        self.encoder6 = Grouped_multi_Attention(c_list[4], c_list[5])

        # Simplified feature fusion modules
        # (each module takes the decoder output plus the skip connection feature as input)
        self.fuse5 = SimpleFuse(c_list[4] * 2, c_list[4])  # d5_out_ch + x5_ch
        self.fuse4 = SimpleFuse(c_list[3] * 2, c_list[3])  # d4_out_ch + x4_ch
        self.fuse3 = SimpleFuse(c_list[2] * 2, c_list[2])  # d3_out_ch + x3_ch
        self.fuse2 = SimpleFuse(c_list[1] * 2, c_list[1])  # d2_out_ch + x2_ch
        self.fuse1 = SimpleFuse(c_list[0] * 2, c_list[0])  # d1_out_ch + x1_ch

        # Decoder
        self.decoder5 = Grouped_multi_Attention(c_list[5], c_list[4])
        self.decoder4 = Grouped_multi_Attention(c_list[4], c_list[3])
        self.decoder3 = Grouped_multi_Attention(c_list[3], c_list[2])
        self.decoder2 = nn.Conv2d(c_list[2], c_list[1], 3, padding=1)
        self.decoder1 = nn.Conv2d(c_list[1], c_list[0], 3, padding=1)

        # Final output layer
        self.final = nn.Conv2d(c_list[0], num_classes, 1)
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Conv2d):
            fan_out = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
            fan_out //= m.groups
            nn.init.normal_(m.weight, 0, math.sqrt(2.0 / fan_out))
            if m.bias is not None:
                nn.init.zeros_(m.bias)

    def forward(self, x):
        # Encoding
        x1 = F.gelu(self.encoder1(x))
        x2 = F.gelu(self.encoder2(F.max_pool2d(x1, 2)))
        x3 = F.gelu(self.encoder3(F.max_pool2d(x2, 2)))
        x4 = F.gelu(self.encoder4(F.max_pool2d(x3, 2)))
        x5 = F.gelu(self.encoder5(F.max_pool2d(x4, 2)))
        x6 = F.gelu(self.encoder6(F.max_pool2d(x5, 2)))

        # Decoding and fusion
        d5 = self.decoder5(x6)
        u5 = self.fuse5(d5, x5)
        d4 = self.decoder4(u5)
        u4 = self.fuse4(d4, x4)
        d3 = self.decoder3(u4)
        u3 = self.fuse3(d3, x3)
        d2 = self.decoder2(F.interpolate(u3, scale_factor=2, mode='bilinear', align_corners=False))
        u2 = self.fuse2(d2, x2)
        d1 = self.decoder1(F.interpolate(u2, scale_factor=2, mode='bilinear', align_corners=False))
        u1 = self.fuse1(d1, x1)

        out = F.interpolate(self.final(u1), scale_factor=2, mode='bilinear', align_corners=False)
        return torch.sigmoid(out)



### 3、model training

In [ ]:
import torch
import cv2
import os
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torch import nn, optim
from torchvision.transforms import transforms
# from unet import Unet
# from dataset import LiverDataset
# from common_tools import transform_invert

val_interval = 2  # Variable that defines how many epochs to wait between validation runs
num_epochs = 20
batch_size = 8

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Define the transform that converts the input image to a tensor
x_transforms = transforms.ToTensor()
y_transforms = transforms.ToTensor()

model = improved_UNet(1, 1).to(device)  # Create a U-Net model with 1 input channel and 1 output channel, then move it to the device (GPU or CPU)
model = nn.DataParallel(model)  # Use DataParallel for single-machine multi-GPU training

train_curve = list()  # Empty list for storing training loss values
valid_curve = list()

liver_dataset = LiverDataset("./data/train/", transform=x_transforms, target_transform=y_transforms)
dataload = DataLoader(liver_dataset, batch_size=batch_size, shuffle=True,num_workers=4)
optimizer = optim.Adam(model.parameters())
criterion = nn.BCEWithLogitsLoss()

for epoch in range(num_epochs):
    print('Epoch {}/{}'.format(epoch, num_epochs))  # Print the current epoch and the total number of epochs
    print('-' * 10)  # Print a separator line
    dt_size = len(dataload.dataset)  # Get the size of the dataset (number of images)
    epoch_loss = 0
    step = 0
    for x, y in dataload:  # Iterate through each batch in the data loader
        step += 1
        inputs = x.to(device)  # Move the input image to the device
        labels = y.to(device)
        # zero the parameter gradients
        optimizer.zero_grad()
        # forward
        outputs = model(inputs)
        loss = criterion(outputs, labels)  # Use the loss function to compute the difference between outputs and labels
        loss.backward()  # Use backpropagation to compute the gradients of the model parameters
        optimizer.step()  # Use the optimizer to update the model parameters
        epoch_loss += loss.item()  # Accumulate the current batch loss into the epoch loss
        train_curve.append(loss.item())  # Append the current batch loss to the training curve list

    print("epoch %d train loss:%0.3f" % (epoch, epoch_loss/step))  # Print the current epoch and the average loss (total loss divided by total steps)
    if epoch == 3:
        torch.save(model.state_dict(), './model/3_weights.pth')


    valid_dataset = LiverDataset("./data/val/", transform=x_transforms, target_transform=y_transforms)
    valid_loader = DataLoader(valid_dataset, batch_size=4, shuffle=True)
    if (epoch) % val_interval == 0: 
        loss_val = 0.
        model.eval()  # Switch the model to evaluation mode (no gradients, no randomness)
        with torch.no_grad():  # Context manager indicating that gradients are not needed
            step_val = 0
            for x, y in valid_loader:
                step_val += 1
                x = x.type(torch.FloatTensor)
                inputs = x.to(device)
                labels = y.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                loss_val += loss.item()

            valid_curve.append(loss_val)
            print("epoch %d valid_loss:%0.3f" % (epoch, loss_val / step_val))  # Print the current epoch and the average validation loss (total validation loss divided by total validation steps)
train_x = range(len(train_curve))
train_y = train_curve
train_iters = len(dataload)
valid_x = np.arange(1, len(
    valid_curve) + 1) * train_iters * val_interval  
valid_y = valid_curve

plt.plot(train_x, train_y, label='Train')
plt.plot(valid_x, valid_y, label='Valid')

plt.legend(loc='upper right')
plt.ylabel('loss value')
plt.xlabel('Iteration')
plt.show()
torch.save(model.state_dict(), './model/test_weights.pth')

### 4、prediction

In [12]:
import os
import cv2
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
from PIL import Image


def transform_invert(img_, transform_train):
    if isinstance(transform_train, transforms.Compose):
        norm_t = [t for t in transform_train.transforms if isinstance(t, transforms.Normalize)]
        if norm_t:
            mean = torch.tensor(norm_t[0].mean, dtype=img_.dtype, device=img_.device)
            std  = torch.tensor(norm_t[0].std,  dtype=img_.dtype, device=img_.device)
            img_ = img_ * std[:,None,None] + mean[:,None,None]
    arr = img_.cpu().detach().numpy().transpose(1,2,0)*255
    arr = arr.astype(np.uint8)
    if arr.shape[2] == 3:
        return Image.fromarray(arr, mode="RGB")
    else:
        return Image.fromarray(arr.squeeze(), mode="L")


def post_process(prob_map, thresh=0.5):
    mask = (prob_map > thresh).astype(np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((5,5),np.uint8))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  np.ones((3,3),np.uint8))
    return mask


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
save_root = "./data/predict"
os.makedirs(save_root, exist_ok=True)

# transforms
x_transforms = transforms.ToTensor()
y_transforms = transforms.ToTensor()

# --------------------------
# Load the model
# --------------------------
model = improved_UNet(num_classes=1, input_channels=1).to(device)
state = torch.load("./model/test_weights_att.pth", map_location=device)
state = {k.replace("module.",""):v for k,v in state.items()}
model.load_state_dict(state)
model.eval()

# --------------------------
# Data loading
# --------------------------
liver_dataset = LiverDataset(
    "./data/val/",
    transform=x_transforms,
    target_transform=y_transforms
)
loader = DataLoader(liver_dataset, batch_size=1, num_workers=0)

plt.ion()
idx = 0
best_thresh = 0.5  # Optimal threshold found during validation

# --------------------------
# Inference and saving
# --------------------------
with torch.no_grad():
    for x, gt in loader:
        x  = x.to(device, dtype=torch.float32)   # [1,1,H,W]
        gt = gt.to(device, dtype=torch.float32)  # [1,1,H,W]

        y = model(x)
        if isinstance(y, tuple):
            y = y[-1]
        # Align spatial dimensions
        if y.shape[2:] != x.shape[2:]:
            y = F.interpolate(y, size=x.shape[2:], mode='bilinear', align_corners=False)

        # Save the original image and the ground truth
        img_x  = transform_invert(x.squeeze(0), x_transforms)
        img_gt = transform_invert(gt.squeeze(0), y_transforms)
        img_x.save(os.path.join(save_root, f"predict_{idx}_s.png"))
        img_gt.save(os.path.join(save_root, f"predict_{idx}_g.png"))

        # Probability map -> post-processing -> binary mask
        prob = y.squeeze().cpu().numpy()  # [H,W]
        mask = post_process(prob, thresh=best_thresh)
        cv2.imwrite(os.path.join(save_root, f"predict_{idx}_o.png"), mask*255)

        print(f"Saved image {idx}")
        idx += 1

plt.ioff()



Saved image 0
Saved image 1
Saved image 2
Saved image 3
Saved image 4
Saved image 5
Saved image 6
Saved image 7
Saved image 8
Saved image 9
Saved image 10
Saved image 11
Saved image 12
Saved image 13
Saved image 14
Saved image 15
Saved image 16
Saved image 17
Saved image 18
Saved image 19
Saved image 20
Saved image 21
Saved image 22
Saved image 23
Saved image 24
Saved image 25
Saved image 26
Saved image 27


### 6、Computing the Dice coefficient of the prediction results

In [14]:
import cv2

root = './data/predict'  # Define the root directory where the results are saved
nums = len(os.listdir(root)) // 3
dice = list()
dice_mean = 0
# Iterate through each image
for i in range(nums):
    ground_path = os.path.join(root, "predict_%d_g.png" % i)
    predict_path = os.path.join(root, "predict_%d_o.png" % i)
    # Use OpenCV to read the output label files and get the images as numpy arrays
    img_ground = cv2.imread(ground_path)
    img_predict = cv2.imread(predict_path)
    intersec = 0 
    x = 0 
    y = 0 
    for w in range(img_ground.shape[0]):  # Loop over each row (width)
        for h in range(img_ground.shape[1]):
            
            intersec += img_ground.item(w, h, 1) * img_predict.item(w, h, 1) / (255 * 255)
            x += img_ground.item(w, h, 1) / 255
            y += img_predict.item(w, h, 1) / 255
    if x + y == 0:  # If both x and y are 0, both images are empty sets
        current_dice = 1  # Set the Dice coefficient to 1 (two empty sets are considered identical)
    else:

        current_dice = round(2 * intersec / (x + y), 3)
    dice_mean += current_dice
    dice.append(current_dice)
dice_mean /= len(dice)
dice_array = np.array(dice)
reshaped = dice_array.reshape(-1, 3)
print(reshaped)
print(round(dice_mean, 3))  # Print the average Dice coefficient over all images

[[0.88  0.926 0.93 ]
 [0.815 0.945 0.894]
 [0.866 0.83  0.828]
 [0.533 0.724 0.863]
 [0.769 0.81  0.915]
 [0.968 0.721 0.903]
 [0.949 0.908 0.831]
 [0.887 0.923 0.82 ]
 [0.834 0.88  0.963]
 [0.864 0.864 0.864]]
0.857
